# v15b: Protected Subgraph — Tight Protection

Drop background entirely. Tighter protection (3.0σ, no dilation) to actually reduce to ~5K-10K nodes.

In [ ]:
import numpy as np
import os, re, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINEConv, BatchNorm
from scipy.ndimage import binary_dilation
import fastloops

DATA_ROOT = "/scratch/ud3d4/acm_data/Data"
RESULTS_DIR = "/home/ud3d4/Desktop/SWOG/results/v15b_tight_subgraph"
os.makedirs(RESULTS_DIR, exist_ok=True)
HU_MIN, HU_MAX = -50, 250
np.random.seed(42); torch.manual_seed(42)
def pr(msg=""): print(msg, flush=True)

def load_and_convert(vid):
    ct = np.load(os.path.join(DATA_ROOT, "ct", f"volume-{vid}.npy")).astype(np.float32)
    seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")).astype(np.int32)
    ct_u8 = np.clip(ct, HU_MIN, HU_MAX)
    ct_u8 = ((ct_u8 - HU_MIN) / (HU_MAX - HU_MIN) * 255).round().astype(np.uint8)
    return ct, seg, np.ascontiguousarray(ct_u8[..., np.newaxis])

def discover_volumes():
    vids = []
    for f in sorted(os.listdir(os.path.join(DATA_ROOT, "ct"))):
        m = re.match(r"volume-(\d+)\.npy", f)
        if m:
            vid = int(m.group(1))
            seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy"))
            if (seg == 2).sum() > 0: vids.append(vid)
    return sorted(vids)

def bbox_from_mask(mask, margin=32):
    coords = np.argwhere(mask)
    lo = np.maximum(coords.min(0) - margin, 0)
    hi = np.minimum(coords.max(0) + 1 + margin, mask.shape)
    return tuple(slice(int(lo[i]), int(hi[i])) for i in range(3))

all_vids = discover_volumes()
perm = np.random.permutation(len(all_vids))
n_tr = int(0.7 * len(all_vids)); n_va = int(0.15 * len(all_vids))
train_ids = sorted([all_vids[i] for i in perm[:n_tr]])
val_ids = sorted([all_vids[i] for i in perm[n_tr:n_tr+n_va]])
test_ids = sorted([all_vids[i] for i in perm[n_tr+n_va:]])
pr(f"Found {len(all_vids)} volumes. Split: {len(train_ids)}/{len(val_ids)}/{len(test_ids)}")

# TIGHT protection: 3.0σ, no dilation, but keep dilate=1 to not miss boundary voxels
PSI, ALPHA = 5, 25
STD_MULT, DILATE = 3.0, 1

def _layout(C=1):
    return dict(area=0, s=[1,2,3], cov=[(4,0,0),(5,1,1),(6,2,2),(7,0,1),(8,0,2),(9,1,2)],
                chan0=10, boundary=10+C+6, D=3)

def node_invariants(nf, C=1, eps=1e-6):
    f = nf.astype(np.float64); L = _layout(C); N = f.shape[0]
    V = f[:, L["area"]]; Vs = np.maximum(V, 1.0)
    mc = np.stack([f[:, c] for c in L["s"]], axis=1) / Vs[:, None]
    cov = np.zeros((N, 3, 3))
    for col, i, j in L["cov"]:
        cij = f[:, col] / Vs - mc[:, i] * mc[:, j]; cov[:, i, j] = cij; cov[:, j, i] = cij
    w, vec = np.linalg.eigh(cov); w = np.clip(w, 0.0, None)
    principal = vec[..., -1]; degen = w.sum(1) < eps; denom = w[:, 2] + eps
    shape = np.stack([(w[:,2]-w[:,1])/denom, (w[:,1]-w[:,0])/denom, w[:,0]/denom], axis=1)
    shape[degen] = 0.0; line_like = np.where(degen, 0.0, shape[:, 0])
    chan = f[:, L["chan0"]:L["chan0"]+C] / Vs[:, None] / 255.0
    comp = f[:, L["boundary"]] / np.power(Vs, 2.0/3.0)
    return dict(V=V, surface=f[:,L["boundary"]], centroid=mc, shape=shape,
                line_like=line_like, principal=principal, chan=chan,
                compactness=comp, degenerate=degen.astype(np.float64))

def make_node_features(nf):
    inv = node_invariants(nf)
    x = np.column_stack([np.log1p(inv["V"]), inv["chan"][:,0],
        inv["shape"][:,0], inv["shape"][:,1], inv["shape"][:,2],
        inv["compactness"], inv["degenerate"]]).astype(np.float32)
    for c in range(x.shape[1]):
        mu, sig = x[:,c].mean(), x[:,c].std()
        if sig > 1e-8: x[:,c] = (x[:,c] - mu) / sig
        else: x[:,c] = 0.0
    return x

def make_edge_features(nf, ei, ef, eps=1e-6):
    inv = node_invariants(nf); a, b = ei[0].astype(np.int64), ei[1].astype(np.int64)
    e = ef.astype(np.float64); bl = np.maximum(e[:,0], 1.0)
    Va, Vb = inv["V"][a], inv["V"][b]
    cols = [
        (np.abs(Va-Vb)/(Va+Vb+eps))[:,None],
        (e[:,0]/(inv["surface"][a]+eps))[:,None],
        (e[:,0]/(inv["surface"][b]+eps))[:,None],
        np.abs(inv["chan"][a]-inv["chan"][b]) / (inv["chan"][a]+inv["chan"][b]+eps),
        np.abs(inv["shape"][a]-inv["shape"][b]),
        (np.abs(np.sum(inv["principal"][a]*inv["principal"][b],axis=1))
         * np.minimum(inv["line_like"][a], inv["line_like"][b]))[:,None],
        ((e[:,1]/bl)/255.0)[:,None],
        np.log(np.maximum(Va,1.0)/np.maximum(Vb,1.0))[:,None],
        np.log(np.maximum(inv["chan"][a],eps)/np.maximum(inv["chan"][b],eps)),
        np.log(np.maximum(inv["compactness"][a],eps)/np.maximum(inv["compactness"][b],eps))[:,None],
    ]
    return np.concatenate(cols, axis=1).astype(np.float32)

pr(f"Functions defined. Protection: std_mult={STD_MULT}, dilate={DILATE}")

In [2]:
def build_protected_subgraph(vid):
    """Build Stage 1 graph, then extract ONLY protected nodes as subgraph."""
    ct_raw, seg, _ = load_and_convert(vid)
    organ_mask = (seg == 1) | (seg == 2)
    slc = bbox_from_mask(organ_mask)
    ct_crop = ct_raw[slc]; seg_crop = seg[slc]; organ_crop = organ_mask[slc]
    ct_u8_crop = np.clip(ct_crop, HU_MIN, HU_MAX)
    ct_u8_crop = ((ct_u8_crop - HU_MIN) / (HU_MAX - HU_MIN) * 255).round().astype(np.uint8)
    ct_u8_crop = np.ascontiguousarray(ct_u8_crop[..., np.newaxis])
    
    # Protection mask
    liver_vals = ct_u8_crop[..., 0][organ_crop]
    candidate = organ_crop & (ct_u8_crop[..., 0] < liver_vals.mean() - STD_MULT * liver_vals.std())
    protect = binary_dilation(candidate, iterations=DILATE).astype(np.uint8)
    
    # Stage 1: coarsen with protection + default deletion
    nf, ei, ef, labels, adj = fastloops.merge_and_cut_protected(
        ct_u8_crop, np.ascontiguousarray(protect),
        merge_distance=PSI, cut_distance=ALPHA, connectivity="faces")
    
    raw_nf = np.asarray(nf); raw_ei = np.asarray(ei); raw_ef = np.asarray(ef)
    labels_np = np.asarray(labels)
    n_total = raw_nf.shape[0]
    
    # Identify which nodes are protected
    flat_labels = labels_np.ravel(); flat_prot = protect.ravel().astype(bool)
    valid = flat_labels >= 0
    max_id = int(flat_labels[valid].max()) if valid.any() else -1
    node_prot = np.zeros(n_total, dtype=bool)
    if max_id >= 0:
        pc = np.bincount(flat_labels[valid & flat_prot], minlength=max_id+1)
        node_prot[:min(n_total, len(pc))] = pc[:n_total] > 0
    
    # Extract subgraph: only protected nodes
    prot_ids = np.where(node_prot)[0]
    if len(prot_ids) == 0:
        return None
    
    # Remap node IDs: old_id -> new_id (only protected nodes)
    old2new = -np.ones(n_total, dtype=np.int64)
    old2new[prot_ids] = np.arange(len(prot_ids))
    
    sub_nf = raw_nf[prot_ids]
    
    # Filter edges: keep only edges where BOTH endpoints are protected
    if raw_ei.shape[1] > 0:
        src, dst = raw_ei[0], raw_ei[1]
        mask = node_prot[src] & node_prot[dst]
        sub_src = old2new[src[mask]]; sub_dst = old2new[dst[mask]]
        sub_ei = np.stack([sub_src, sub_dst])
        sub_ef = raw_ef[mask]
    else:
        sub_ei = np.zeros((2, 0), dtype=np.int64)
        sub_ef = np.zeros((0, 4), dtype=np.uint64)
    
    # Build node features
    x = make_node_features(sub_nf)
    
    # Build edge features (undirected: double)
    if sub_ei.shape[1] > 0:
        ea = make_edge_features(sub_nf, sub_ei, sub_ef)
        ei_fwd = torch.tensor(sub_ei, dtype=torch.long)
        ei_rev = torch.stack([ei_fwd[1], ei_fwd[0]])
        edge_index = torch.cat([ei_fwd, ei_rev], dim=1)
        edge_attr = torch.tensor(np.concatenate([ea, ea]), dtype=torch.float32)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 10), dtype=torch.float32)
    
    # Labels from GT
    gt = (seg_crop.ravel() == 2).astype(np.float64)
    n_sub = len(prot_ids)
    n_fg = np.zeros(n_sub, dtype=np.float64)
    n_bg = np.zeros(n_sub, dtype=np.float64)
    if valid.any():
        ofg = np.bincount(flat_labels[valid], weights=gt[valid], minlength=max_id+1)
        otot = np.bincount(flat_labels[valid], minlength=max_id+1)
        obg = otot - ofg
        for new_i, old_i in enumerate(prot_ids):
            if old_i < len(ofg):
                n_fg[new_i] = ofg[old_i]; n_bg[new_i] = obg[old_i]
    
    overlap = n_fg / np.maximum(n_fg + n_bg, 1)
    y = (overlap >= 0.10).astype(np.int64)
    
    data = Data(x=torch.tensor(x, dtype=torch.float32),
                edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor(y, dtype=torch.long),
                n_fg=torch.tensor(n_fg, dtype=torch.float32),
                n_bg=torch.tensor(n_bg, dtype=torch.float32))
    
    # For voxel lifting: need mapping from protected node -> voxels
    # Store: original labels_np, prot_ids (protected original node IDs), seg_crop
    return data, labels_np, prot_ids, seg_crop, n_total

# Build all graphs
pr(f"Building protected subgraphs for {len(all_vids)} volumes...")
graphs = {}; meta = {}
t0 = time.time()
for vid in train_ids + val_ids + test_ids:
    result = build_protected_subgraph(vid)
    if result is None:
        pr(f"  vol-{vid}: SKIP (no protected nodes)")
        continue
    data, labels_np, prot_ids, seg_crop, n_total = result
    graphs[vid] = data
    meta[vid] = dict(labels=labels_np, prot_ids=prot_ids, seg=seg_crop, n_total=n_total)
    
    split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
    n_tu = int((data.y == 1).sum()); n_bg = int((data.y == 0).sum())
    tu_frac = n_tu / max(n_tu + n_bg, 1) * 100
    pr(f"  vol-{vid:>3d} [{split:>5s}]: {data.num_nodes:>6,} nodes ({n_tu:>4,} tu {tu_frac:4.1f}%) "
       f"{data.num_edges:>7,} edges  (was {n_total:>7,} total)")

pr(f"\nBuilt {len(graphs)} graphs in {time.time()-t0:.1f}s")
nodes = [g.num_nodes for g in graphs.values()]
pr(f"Nodes: mean={np.mean(nodes):,.0f} median={np.median(nodes):,.0f} "
   f"max={max(nodes):,} min={min(nodes):,}")
tu_fracs = [float((g.y==1).sum()) / max(g.num_nodes, 1) * 100 for g in graphs.values()]
pr(f"Tumor fraction: mean={np.mean(tu_fracs):.1f}% median={np.median(tu_fracs):.1f}%")

Building protected subgraphs for 118 volumes...


  vol-  0 [train]: 24,851 nodes ( 269 tu  1.1%) 144,638 edges  (was  25,572 total)


  vol-  3 [train]: 213,056 nodes ( 287 tu  0.1%) 1,130,588 edges  (was 218,910 total)


  vol-  4 [train]: 211,460 nodes (128,950 tu 61.0%) 1,448,442 edges  (was 218,726 total)


  vol-  5 [train]: 151,816 nodes (  59 tu  0.0%) 832,686 edges  (was 153,152 total)


...

  vol-126 [ test]: 216,302 nodes ( 633 tu  0.3%) 1,181,596 edges  (was 219,032 total)


  vol-130 [ test]: 308,182 nodes (127,995 tu 41.5%) 1,827,384 edges  (was 316,199 total)



Built 118 graphs in 160.7s


Nodes: mean=168,295 median=128,050 max=881,768 min=15,882


Tumor fraction: mean=7.8% median=1.8%


In [3]:
# ---- GINE + Training ----
class GINE(nn.Module):
    def __init__(self, nd, ed, h=128):
        super().__init__()
        self.ep = nn.Linear(ed, h)
        def mlp(d): return nn.Sequential(nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Linear(h, h))
        self.c1 = GINEConv(mlp(nd), edge_dim=h); self.b1 = BatchNorm(h)
        self.c2 = GINEConv(mlp(h), edge_dim=h); self.b2 = BatchNorm(h)
        self.c3 = GINEConv(mlp(h), edge_dim=h); self.b3 = BatchNorm(h)
        self.head = nn.Linear(h, 2)
    def forward(self, x, ei, ea):
        if ea is not None and ea.numel() > 0: ea = self.ep(ea)
        else:
            n = x.size(0); ei = torch.stack([torch.arange(n, device=x.device)]*2)
            ea = torch.zeros(n, self.ep.out_features, device=x.device)
        x = F.relu(self.b1(self.c1(x, ei, ea)))
        x = F.relu(self.b2(self.c2(x, ei, ea)))
        x = F.relu(self.b3(self.c3(x, ei, ea)))
        return self.head(x)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
trainable = [v for v in train_ids if v in graphs]
val_usable = [v for v in val_ids if v in graphs]
pr(f"Device: {device}")
pr(f"Trainable: {len(trainable)}/{len(train_ids)}, Val: {len(val_usable)}/{len(val_ids)}")

total_pos = sum(int((graphs[v].y==1).sum()) for v in trainable)
total_neg = sum(int((graphs[v].y==0).sum()) for v in trainable)
ratio = total_neg / max(total_pos, 1)
eff = min(np.sqrt(ratio), 30.0)
cw = torch.tensor([1.0, eff], dtype=torch.float32).to(device)
pr(f"Class weight: [1.0, {eff:.1f}] (pos:neg = {total_pos}:{total_neg}, ratio {ratio:.0f}:1)")

nd = graphs[trainable[0]].x.shape[1]
ed = graphs[trainable[0]].edge_attr.shape[1] if graphs[trainable[0]].edge_attr.numel() > 0 else 10
model = GINE(nd, ed).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

PATIENCE, EPOCHS, VAL_EVERY = 15, 200, 3
best_dice, best_state, wait = -1.0, None, 0
history = {"train_loss": [], "val_dice": []}

for epoch in range(1, EPOCHS+1):
    model.train(); eloss = 0.0; proc = 0
    for vid in np.random.permutation(trainable):
        try:
            g = graphs[vid].to(device); opt.zero_grad()
            logits = model(g.x, g.edge_index, g.edge_attr)
            loss = F.cross_entropy(logits, g.y, weight=cw)
            loss.backward(); opt.step(); eloss += loss.item(); proc += 1
            del g, logits, loss
        except torch.cuda.OutOfMemoryError:
            try: del g
            except: pass
            torch.cuda.empty_cache(); continue
        torch.cuda.empty_cache()
    if proc == 0: pr("All OOM"); break
    ml = eloss / proc; history["train_loss"].append(ml)
    
    if epoch % VAL_EVERY == 0 or epoch <= 3:
        model.eval(); tp = fp = fn = 0
        with torch.no_grad():
            for vid in val_usable:
                g = graphs[vid]; m = meta[vid]
                try:
                    gd = g.to(device)
                    preds = model(gd.x, gd.edge_index, gd.edge_attr).argmax(1).cpu().numpy()
                    del gd; torch.cuda.empty_cache()
                except:
                    torch.cuda.empty_cache()
                    preds = model.cpu()(g.x, g.edge_index, g.edge_attr).argmax(1).numpy()
                    model.to(device)
                # Lift: protected subgraph node -> original supernode -> voxel
                labels_np = m["labels"]; prot_ids = m["prot_ids"]; seg_crop = m["seg"]
                flat = labels_np.ravel(); valid = flat >= 0
                if not valid.any(): continue
                mid = int(flat[valid].max())
                # Map: original node ID -> prediction (0 for non-protected nodes)
                lut = np.zeros(mid + 1, dtype=np.int8)
                for new_i, old_i in enumerate(prot_ids):
                    if old_i <= mid and new_i < len(preds):
                        lut[old_i] = preds[new_i]
                pm = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)
                gm = seg_crop == 2; inter = int((pm & gm).sum())
                tp += inter; fp += int(pm.sum()) - inter; fn += int(gm.sum()) - inter
        
        vd = 2*tp/(2*tp+fp+fn+1e-8); history["val_dice"].append(vd)
        if vd > best_dice:
            best_dice = vd; best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
            wait = 0; mk = " *"
        else: wait += 1; mk = ""
        pr(f"  Epoch {epoch:3d}  loss={ml:.4f}  val_dice={vd:.4f}  ({proc} vols){mk}")
        if wait >= PATIENCE: pr(f"  Early stop, best={best_dice:.4f}"); break
    else: pr(f"  Epoch {epoch:3d}  loss={ml:.4f}  ({proc} vols)")

if best_state: model.load_state_dict(best_state)
pr(f"\nBest val Dice: {best_dice:.4f}")
torch.save(best_state or model.state_dict(), os.path.join(RESULTS_DIR, "model.pt"))

Device: cuda:0


Trainable: 82/82, Val: 17/17


Class weight: [1.0, 3.6] (pos:neg = 971775:12521375, ratio 13:1)


  Epoch   1  loss=0.3993  val_dice=0.4125  (82 vols) *


  Epoch   2  loss=0.3382  val_dice=0.2071  (82 vols)


...

  Epoch  67  loss=0.1458  (82 vols)


  Epoch  68  loss=0.1911  (82 vols)


  Epoch  69  loss=0.1567  val_dice=0.4768  (82 vols)


  Early stop, best=0.4951



Best val Dice: 0.4951


In [4]:
# ---- Final evaluation ----
model.eval(); results = []
for vid in sorted(graphs.keys()):
    g = graphs[vid]; m = meta[vid]
    with torch.no_grad():
        try:
            gd = g.to(device)
            preds = model(gd.x, gd.edge_index, gd.edge_attr).argmax(1).cpu().numpy()
            del gd; torch.cuda.empty_cache()
        except:
            torch.cuda.empty_cache()
            preds = model.cpu()(g.x, g.edge_index, g.edge_attr).argmax(1).numpy()
            model.to(device)
    labels_np = m["labels"]; prot_ids = m["prot_ids"]; seg_crop = m["seg"]
    flat = labels_np.ravel(); valid = flat >= 0
    pm = np.zeros(labels_np.shape, dtype=bool)
    if valid.any():
        mid = int(flat[valid].max())
        lut = np.zeros(mid+1, dtype=np.int8)
        for new_i, old_i in enumerate(prot_ids):
            if old_i <= mid and new_i < len(preds): lut[old_i] = preds[new_i]
        pm = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)
    gm = seg_crop == 2; inter = int((gm & pm).sum())
    dice = 2.0*inter/(gm.sum()+pm.sum()+1e-8)
    rec = inter/(gm.sum()+1e-8)
    prec = inter/(pm.sum()+1e-8) if pm.sum()>0 else 0.0
    fg = g.n_fg.cpu().numpy(); bg = g.n_bg.cpu().numpy()
    gt_t = fg.sum(); maj = fg > bg
    oracle = float(2*fg[maj].sum()/(2*fg[maj].sum()+bg[maj].sum()+(gt_t-fg[maj].sum())+1e-8)) if gt_t > 0 else 0.0
    split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
    results.append(dict(vid=vid, split=split, dice=float(dice), recall=float(rec),
                        precision=float(prec), oracle=oracle, nodes=g.num_nodes,
                        tumor_nodes=int((g.y==1).sum())))

pr(f"\n{'='*80}")
pr(f"  v15 RESULTS: Protected Subgraph Only")
pr(f"{'='*80}")
pr(f"  Params: psi={PSI} alpha={ALPHA} std_mult={STD_MULT} dilate={DILATE}")
pr(f"  {'Split':>5s}  {'Dice':>8s}  {'Recall':>8s}  {'Prec':>8s}  {'Oracle':>8s}  {'Nodes':>7s}  {'N':>3s}")
pr(f"  {'-'*52}")
for split in ["train", "val", "test"]:
    s = [r for r in results if r["split"] == split]
    if s:
        pr(f"  {split:>5s}  {np.mean([r['dice'] for r in s]):8.4f}  "
           f"{np.mean([r['recall'] for r in s]):8.4f}  "
           f"{np.mean([r['precision'] for r in s]):8.4f}  "
           f"{np.mean([r['oracle'] for r in s]):8.4f}  "
           f"{np.mean([r['nodes'] for r in s]):7.0f}  {len(s):3d}")

pr(f"\n  Best val Dice: {best_dice:.4f}")
pr(f"  Paper target: 0.891")
pr(f"  Previous best (v11, 172K full graph): val 0.48, test 0.32")

with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(dict(params=dict(psi=PSI, alpha=ALPHA, std_mult=STD_MULT, dilate=DILATE),
                   best_val_dice=float(best_dice), epochs=len(history["train_loss"]),
                   history=history, results=results), f, indent=2)
pr(f"  Saved to {RESULTS_DIR}/")

  v15 RESULTS: Protected Subgraph Only


  Params: psi=5 alpha=25 std_mult=1.5 dilate=2


  Split      Dice    Recall      Prec    Oracle    Nodes    N


  ----------------------------------------------------


  train    0.3127    0.4010    0.3710    0.9242   164551   82


...

   test    0.3411    0.4949    0.4295    0.9348   189046   19



  Best val Dice: 0.4951


  Paper target: 0.891


  Previous best (v11, 172K full graph): val 0.48, test 0.32


  Saved to /home/ud3d4/Desktop/SWOG/results/v15_protected_subgraph/
